Pandas Part 1 — Exercises

Section 1 — Bank Dataset

In [1]:
import pandas as pd

In [2]:
import numpy as np

In [3]:
df = pd.read_csv("../data/bank_full.csv", sep=";")
df


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,51,technician,married,tertiary,no,825,no,no,cellular,17,nov,977,3,-1,0,unknown,yes
45207,71,retired,divorced,primary,no,1729,no,no,cellular,17,nov,456,2,-1,0,unknown,yes
45208,72,retired,married,secondary,no,5715,no,no,cellular,17,nov,1127,5,184,3,success,yes
45209,57,blue-collar,married,secondary,no,668,no,no,telephone,17,nov,508,4,-1,0,unknown,no


1\. Looking only at the group of divorced individuals, what percentage of them are entrepreneurs? And students?

In [4]:
divorced = df[df['marital'] == 'divorced']

percentages = divorced['job'].value_counts(normalize=True) * 100

percentages[['entrepreneur', 'student']]

job
entrepreneur    3.437680
student         0.115229
Name: proportion, dtype: float64

2\. Your manager decided to create two categories, Premium and Gold, to focus on the most promising clients.

For the Premium category, include those whose balance is in the top 10% — that is, above the 90th percentile.

For the Gold group, include those whose balance is in the top 20%, but below the Premium group — that is, from the 80th percentile up to (but not including) the 90th percentile.

Check how many clients fall into the Premium and Gold groups.

In [5]:
p90 = df['balance'].quantile(0.9)
p80 = df['balance'].quantile(0.8)


premium = df[df['balance'] > p90]
gold = df[(df["balance"] > p80) & (df["balance"] < p90)]


print("Premium:", len(premium))
print("Gold:", len(gold))

Premium: 4521
Gold: 4518


3\. Your boss also wants to know if there is any difference in the percentages observed in exercise (1) for the Premium group. Check how the cross table of the overall population differs from the Premium group. Is there a difference? What is your hypothesis for why this difference exists?”

In [6]:
divorced = df[df['marital'] == 'divorced']
percentages = divorced['job'].value_counts(normalize=True) * 100

divorced_premium = premium[premium['marital'] == 'divorced']
percentages_premium = divorced_premium['job'].value_counts(normalize=True) * 100

percentages.reindex(['entrepreneur', 'student'], fill_value=0)
percentages_premium.reindex(['entrepreneur', 'student'], fill_value=0)

print("Overall population:")
print(percentages.reindex(['entrepreneur', 'student'], fill_value=0))

print("\nPremium:")
print(percentages_premium.reindex(['entrepreneur', 'student'], fill_value=0))

Overall population:
job
entrepreneur    3.437680
student         0.115229
Name: proportion, dtype: float64

Premium:
job
entrepreneur    2.745995
student         0.000000
Name: proportion, dtype: float64


 ### Q3 -- There is a small difference between the Premium and overall groups, with a slightly lower percentage of entrepreneurs in Premium. However, the difference is minor and may not be significant, especially given the small number of observations in this subset.

4\. For the final presentation, you decide to create a table with the 10 clients who have the highest balance. Since the table is too large, you choose to display only the relevant information: age, job, marital status, education, default, loan, and duration.

Additionally, you decide to highlight clients who were more and less attentive, as they seem to have the highest and lowest potential, respectively. To do this, highlight in yellow the client with the highest duration, and highlight in red the one with the lowest duration.

In [7]:
top10 = df.sort_values(by='balance', ascending=False).head(10)
top10 = top10[['age', 'job', 'marital', 'education', 'default', 'loan', 'duration']]

max_duration = top10['duration'].max()
min_duration = top10['duration'].min()

def highlight_duration(col):
    return [
        'background-color: yellow' if v == col.max() else
        'background-color: red' if v == col.min() else
        ''
        for v in col
    ]




styled_top10 = top10.style.apply(highlight_duration, subset=['duration'])
styled_top10

,age,job,marital,education,default,loan,duration
39989,51,management,single,tertiary,no,no,90
26227,59,management,married,tertiary,no,no,145
43393,84,retired,married,secondary,no,no,390
42558,84,retired,married,secondary,no,no,679
41693,60,retired,married,primary,no,no,205
19785,56,management,divorced,tertiary,no,no,442
21192,52,blue-collar,married,primary,no,no,109
19420,59,admin.,married,unknown,no,no,45
41374,32,entrepreneur,single,tertiary,no,no,69
12926,56,blue-collar,married,secondary,no,no,339


5\. In your opinion, and according to part of the board, there are 3 factors that determine whether a person will accept your bank’s offer (y): balance, whether they have previously defaulted on a loan (default), and whether they have a personal loan (loan).

To evaluate these 3 characteristics together with acceptance (y), you will need to create a pivot table where the observed value is the average balance.

Evaluate whether there is a difference in the balance of people who accept the offer and have a loan versus those who do not accept and do not have a loan, and do the same for default.

In other words, create a pivot table crossing loan and default, while also segmenting by whether the person accepted the offer or not.

Is the balance of those who accept and have a loan higher or lower than those who do not accept? Why do you think this happens? Create your hypothesis to justify the result found.

Hint: Create a pivot table where the index is the columns y and default or loan, the columns field is loan or default, and the values are the average balance (balance).

In [10]:
df.pivot_table(
    index = ["y", "loan"],
    columns = "default",
    values = "balance",
    aggfunc = "mean"
)



default            no         yes
y   loan                         
no  no    1435.673581 -124.737945
    yes    807.983318 -172.958042
yes no    1912.359060  -82.108108
    yes    912.245203  -10.666667

 ### Q5 -- 	Those who accepted the offer and already had a loan have a higher average balance than those who did not accept, with a difference of a little over 100. This may suggest that individuals with higher balances are more financially active or more comfortable taking on additional financial products